# 真实股票数据预测 + 回测收益曲线模板

这个 Notebook 演示一个完整流程：

1. 下载真实股票数据（`yfinance`）
2. 构建特征并预测次日收益率
3. 生成交易信号并进行回测（含交易成本）
4. 输出收益曲线、回撤曲线与关键绩效指标

> 仅用于学习与研究，不构成投资建议。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8')
pd.set_option('display.float_format', lambda x: f"{x:,.6f}")

# ========== 可调参数 ==========
TICKER = 'AAPL'         # 例如: AAPL, TSLA, NVDA, 0700.HK
START_DATE = '2018-01-01'
END_DATE = '2026-01-01'
TRAIN_RATIO = 0.8
RANDOM_STATE = 42

# 交易参数
FEE_BPS = 5             # 单边手续费（基点），5 = 0.05%
PRED_THRESHOLD = 0.0005 # 预测收益率超过该阈值才开仓（过滤噪声）
ANNUALIZATION = 252

print('参数已加载。')

In [ ]:
def download_price_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f'无法下载数据: {ticker}')

    # yfinance 在某些版本下可能返回多层列名
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.rename(columns=str.lower)
    keep_cols = [c for c in ['open', 'high', 'low', 'close', 'volume'] if c in df.columns]
    return df[keep_cols].copy()


raw_df = download_price_data(TICKER, START_DATE, END_DATE)
raw_df.tail()

In [ ]:
def build_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    feat = df.copy()

    feat['ret_1'] = feat['close'].pct_change(1)
    feat['ret_5'] = feat['close'].pct_change(5)
    feat['ret_10'] = feat['close'].pct_change(10)

    feat['ma_5_dev'] = feat['close'] / feat['close'].rolling(5).mean() - 1
    feat['ma_20_dev'] = feat['close'] / feat['close'].rolling(20).mean() - 1
    feat['vol_20'] = feat['ret_1'].rolling(20).std()

    if 'volume' in feat.columns:
        feat['volume_chg_1'] = feat['volume'].pct_change(1)
    else:
        feat['volume_chg_1'] = 0.0

    # 目标：次日收益率
    feat['target'] = feat['close'].pct_change().shift(-1)
    feat = feat.dropna().copy()

    feature_cols = ['ret_1', 'ret_5', 'ret_10', 'ma_5_dev', 'ma_20_dev', 'vol_20', 'volume_chg_1']
    return feat, feature_cols


df, feature_cols = build_features(raw_df)
df[feature_cols + ['target']].tail()

In [ ]:
split_idx = int(len(df) * TRAIN_RATIO)

X_train = df[feature_cols].iloc[:split_idx]
y_train = df['target'].iloc[:split_idx]
X_test = df[feature_cols].iloc[split_idx:]
y_test = df['target'].iloc[split_idx:]

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)
pred = pd.Series(model.predict(X_test), index=X_test.index, name='pred_ret')

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
direction_acc = (np.sign(pred) == np.sign(y_test)).mean()

print(f'Ticker: {TICKER}')
print(f'Train/Test: {len(X_train)} / {len(X_test)}')
print(f'RMSE: {rmse:.6f}')
print(f'MAE: {mae:.6f}')
print(f'方向准确率: {direction_acc:.4f}')

In [ ]:
def annual_return(r: pd.Series, periods_per_year: int = 252) -> float:
    if len(r) == 0:
        return np.nan
    return float((1 + r).prod() ** (periods_per_year / len(r)) - 1)


def sharpe_ratio(r: pd.Series, periods_per_year: int = 252) -> float:
    std = r.std()
    if std == 0 or np.isnan(std):
        return np.nan
    return float(np.sqrt(periods_per_year) * r.mean() / std)


def max_drawdown(equity_curve: pd.Series) -> float:
    rolling_peak = equity_curve.cummax()
    drawdown = equity_curve / rolling_peak - 1
    return float(drawdown.min())


def build_backtest_frame(y_true: pd.Series, y_pred: pd.Series, threshold: float, fee_bps: float) -> pd.DataFrame:
    bt = pd.DataFrame(index=y_true.index)
    bt['true_ret'] = y_true
    bt['pred_ret'] = y_pred

    # 多头策略：预测收益高于阈值则持仓 1，否则空仓 0
    bt['position'] = (bt['pred_ret'] > threshold).astype(float)

    # 当天调仓，成本按仓位变化计算
    bt['turnover'] = bt['position'].diff().abs().fillna(bt['position'].abs())
    fee_rate = fee_bps / 10000.0
    bt['fee'] = bt['turnover'] * fee_rate

    bt['strategy_ret_gross'] = bt['position'] * bt['true_ret']
    bt['strategy_ret_net'] = bt['strategy_ret_gross'] - bt['fee']
    bt['buyhold_ret'] = bt['true_ret']

    bt['equity_strategy'] = (1 + bt['strategy_ret_net']).cumprod()
    bt['equity_buyhold'] = (1 + bt['buyhold_ret']).cumprod()

    bt['drawdown_strategy'] = bt['equity_strategy'] / bt['equity_strategy'].cummax() - 1
    bt['drawdown_buyhold'] = bt['equity_buyhold'] / bt['equity_buyhold'].cummax() - 1
    return bt


bt = build_backtest_frame(y_test, pred, PRED_THRESHOLD, FEE_BPS)
bt.tail()

In [ ]:
def summarize_performance(bt: pd.DataFrame, annualization: int = 252) -> pd.DataFrame:
    strat = bt['strategy_ret_net']
    hold = bt['buyhold_ret']

    summary = pd.DataFrame(
        {
            'Strategy': [
                annual_return(strat, annualization),
                sharpe_ratio(strat, annualization),
                max_drawdown(bt['equity_strategy']),
                float((strat > 0).mean()),
                float(bt['turnover'].mean() * annualization),
                float(bt['equity_strategy'].iloc[-1] - 1),
            ],
            'Buy&Hold': [
                annual_return(hold, annualization),
                sharpe_ratio(hold, annualization),
                max_drawdown(bt['equity_buyhold']),
                float((hold > 0).mean()),
                np.nan,
                float(bt['equity_buyhold'].iloc[-1] - 1),
            ],
        },
        index=['年化收益', 'Sharpe', '最大回撤', '胜率', '年化换手', '累计收益'],
    )
    return summary


summary = summarize_performance(bt, ANNUALIZATION)
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 收益曲线
axes[0].plot(bt.index, bt['equity_buyhold'], label='Buy&Hold', linewidth=1.8)
axes[0].plot(bt.index, bt['equity_strategy'], label='Strategy (Net)', linewidth=1.8)
axes[0].set_title(f'{TICKER} 回测收益曲线（测试集）')
axes[0].set_ylabel('Net Value')
axes[0].legend(loc='upper left')
axes[0].grid(alpha=0.3)

# 回撤曲线
axes[1].plot(bt.index, bt['drawdown_buyhold'], label='Buy&Hold DD', linewidth=1.5)
axes[1].plot(bt.index, bt['drawdown_strategy'], label='Strategy DD', linewidth=1.5)
axes[1].set_title('回撤曲线')
axes[1].set_ylabel('Drawdown')
axes[1].set_xlabel('Date')
axes[1].legend(loc='lower left')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()